# M6 pgvector 기준 검색

BM25 기준과 나란히 볼 수 있도록 고정된 다자료 평가 질문 21개로 PostgreSQL/pgvector 의미 검색 결과를 확인한다. 평가 질문, 정답 문서 ID, 문서 청크 벡터는 기존 M6·Chroma 자료를 그대로 쓴다.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
EVAL_DIR = ROOT / "data/personal/evaluation"
FIGURE_PATH = ROOT / "data/personal/figures/pgvector_hit_rate_by_source.png"

details = pd.read_csv(EVAL_DIR / "pgvector_details.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
summary = pd.read_csv(EVAL_DIR / "pgvector_summary.csv", encoding="utf-8-sig")
meta = json.loads((EVAL_DIR / "pgvector_meta.json").read_text(encoding="utf-8"))
questions_path = EVAL_DIR / "questions_multisource.csv"
question_sha256 = hashlib.sha256(questions_path.read_bytes()).hexdigest()

print(f"평가 질문: {details['qid'].nunique()}개 / 검색 결과: {len(details)}건")
print(f"문서 청크: {meta['chunks_loaded']:,}개 / 고유 문서: {meta['unique_documents_loaded']:,}개")
print(f"임베딩: {meta['embedding_model']} ({meta['dimensions']}차원)")
print(f"질문 파일 해시 일치: {question_sha256 == meta['questions_sha256']}")
print(f"검색 방식: {meta['search_mode']} / 임베딩·LLM API 호출: {meta['embedding_or_llm_api_calls']}회")

assert len(details) == 42 and details['qid'].nunique() == 21
assert question_sha256 == meta['questions_sha256']
assert meta['embedding_or_llm_api_calls'] == 0

In [ ]:
# 자료 유형별 점수와 전체 점수를 함께 본다.
score_view = summary.copy()
for column in ["hit1", "hit5", "precision5", "mrr5"]:
    score_view[column] = score_view[column].map(lambda value: f"{value:.3f}")
display(score_view.sort_values(["scope", "source_type"], ascending=[True, True]).reset_index(drop=True))

In [ ]:
# 전체 자료 검색과 자료 유형 필터 검색의 Hit@1·Hit@5를 나눠 그린다.
source_order = ["kosha_guide", "moel_report", "sif"]
source_labels = ["KOSHA Guide", "MOEL report", "SIF"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, scope, title in zip(
    axes,
    ["all_sources", "source_filtered"],
    ["All sources", "Source filtered"],
):
    part = summary[(summary["scope"] == scope) & (summary["source_type"] != "ALL")].set_index("source_type").reindex(source_order)
    positions = range(len(source_order))
    width = 0.34
    bars1 = ax.bar([p - width/2 for p in positions], part["hit1"], width, label="Hit@1", color="#3567A5")
    bars5 = ax.bar([p + width/2 for p in positions], part["hit5"], width, label="Hit@5", color="#42A58A")
    ax.set_title(title)
    ax.set_xticks(list(positions), source_labels, rotation=12)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Question fraction")
    ax.grid(axis="y", alpha=0.25)
    ax.bar_label(bars1, fmt="%.2f", padding=2, fontsize=8)
    ax.bar_label(bars5, fmt="%.2f", padding=2, fontsize=8)

axes[0].legend(frameon=False)
fig.suptitle("pgvector retrieval quality by source")
fig.tight_layout()
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
display(fig)
print(f"그림 저장: {FIGURE_PATH.relative_to(ROOT)}")

In [ ]:
# 전체 자료 검색에서 정답 문서가 상위 5개에 들어오지 않은 질문을 확인한다.
misses = details[(details["scope"] == "all_sources") & (details["hit5"] == "0")]
miss_view = misses[["qid", "source_type", "query", "gold_doc_ids", "retrieved_doc_ids"]].reset_index(drop=True)
print(f"전체 자료 검색 Hit@5 누락: {len(miss_view)} / {details[details['scope'] == 'all_sources']['qid'].nunique()}개 질문")
display(miss_view)

# 정답은 상위 5개에 있지만 첫 번째 결과는 아닌 질문도 분리한다.
top1_misses = details[(details["scope"] == "all_sources") & (details["hit1"] == "0") & (details["hit5"] == "1")]
print(f"Hit@5에는 포함됐지만 Hit@1은 아닌 질문: {len(top1_misses)}개")
display(top1_misses[["qid", "source_type", "gold_doc_ids", "retrieved_doc_ids"]].reset_index(drop=True))

## pgvector 기준 검색 결과와 재현 조건

- 21개 질문의 전체 자료 검색에서 Hit@5 16/21(0.762), Hit@1 11/21(0.524), Precision@5 0.152, MRR@5 0.613이었다. 청크를 순위화한 뒤 문서 ID 중복을 제거해 고유 문서 상위 5개를 평가했다.
- 기록상 이 실험은 이미 만들어 둔 Chroma 임베딩을 PostgreSQL pgvector로 복사해 정확 코사인 거리 검색을 수행했고, 임베딩·LLM API를 다시 부르지 않았다. ANN 인덱스가 아닌 전체 벡터 비교라 이 결과를 대규모 서비스의 성능 결과로 일반화할 수 없다.
- 비교 당시 PostgreSQL 16.15, pgvector 0.6.0으로 기록되어 있다. 현재 환경에서 DB를 다시 열어 실시간 재현한 결과는 아니다. DSN은 보안상 기록하지 않는다.
- 의미 검색은 표현이 달라도 찾을 수 있지만 질문 임베딩 모델·색인 코퍼스가 바뀌면 순위와 점수가 달라진다. 코퍼스 SHA-256과 질문 해시를 유지해 비교한다.
